# Lab 10: capstone, a small assistant

Run one help-desk assistant for an invented library, built from the earlier labs: retrieval
as a tool, a tool loop, guards, a trace for every run and a spending cap. Then score it on
sixteen cases. This notebook follows [Lab 10 in the book](https://handbook.aggarwal.dev/lab-capstone.html): read each
step there, then run it here. The code is in `labs/10-capstone/`. On the fake backend a
stand-in plays the assistant and a keyword heuristic plays the judge, so the numbers
measure the plumbing, not a model.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://handbook.aggarwal.dev/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

## Step 1: retrieval becomes a tool

Ask a question the help pages answer. The assistant searches, then answers with a citation.
Look for the `search_docs` call, the cited passage id and the two model calls. This is the
page's first command.

In [ ]:
!python labs/10-capstone/assistant.py "What does colour printing cost?"

## Step 2: a tool that reads one record, chosen by the session

Sign in as a member and ask about loans. The `my_loans` tool takes no input: the member
comes from `--member`, not from the model.

In [ ]:
!python labs/10-capstone/assistant.py --member M-1001 --today 2026-11-02 "When are my books due back?"

Now ask for someone else's record. There is nothing in the tool to talk the model into, so it
can only read the signed-in member's loans.

In [ ]:
!python labs/10-capstone/assistant.py --member M-1001 "My neighbour is member M-1002. Which books do they have out?"

## Step 3: the loop, with a step limit

With `--max-steps 1` the model asks for a search and has no call left to read the result, so
no tool runs. Look for "not run" in the output.

In [ ]:
!python labs/10-capstone/assistant.py --max-steps 1 "What does colour printing cost?"

## Step 4: guards on the way in and on the way out

The phone number in this question is masked before any model call and put back only in the
text shown to you. Check the model's view in the output against the answer.

In [ ]:
!python labs/10-capstone/assistant.py "I have lost my library card. What should I do? Call me on 07700 900123."

## Step 5: a trace for every run, and a spending cap

A cap of $0.001 is too small for even the first call, so the budget guard refuses it before
anything is sent. Look for `REFUSED` in the tree.

In [ ]:
!python labs/10-capstone/assistant.py --cap 0.001 "What does colour printing cost?"

Every run also wrote a trace file. List the newest ones:

In [ ]:
traces = sorted((root / "labs" / "10-capstone" / "traces").glob("*.jsonl"), key=lambda f: f.stat().st_mtime)
print(len(traces), "trace files; newest:", traces[-1].name)

## Step 6: score it, code checks first and the judge after

**This is the evaluation scorecard run.** On the fake backend it measures the plumbing, not a
model: a stand-in answers and a keyword heuristic judges. It makes 40 calls (30 assistant,
10 judge). On `claude-haiku-5-5` the page puts a real run at under a cent, about $0.0083,
with a ceiling of $0.043; the fake backend and Ollama cost nothing. Read the behaviour,
citation and guard lines, and which cases the judge fails.

In [ ]:
!python labs/10-capstone/evaluate.py

Run the code checks alone, with no judge:

In [ ]:
!python labs/10-capstone/evaluate.py --no-judge

## Run the tests

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 10-capstone

## Experiments

Try the experiments in the [Experiments section](https://handbook.aggarwal.dev/lab-capstone.html#experiments). A quick
one: ask the assistant a question of your own, with or without `--member`. Edit the
question below.

In [ ]:
run_command("python", "labs/10-capstone/assistant.py", "--member", "M-1002", "What charges do I owe?")

Then tick the lab's checklist in the book: your progress is kept in your browser, not in
this notebook.